# meeting_extracted_w2_t5_export — W2 T5：關卡彙總 ＋ PM 審核例外清單（規則，無 LLM）

**做什麼**：讀 T4 寫好的對照格（`v_meeting_sb_reconcile_q`）與 T3 標註（`v_meeting_items_sb_q`），對一期 `brief_month`：
(1) 算關卡：格數、relation 分布、llm 格沒摘要、shareable 驗證失敗、待審數、prompt 版本是否混雜、相關條目是否都進了格；
(2) 印 PM 例外清單 md（規格 §4.6）：每 app 一節，conflict → supplement → meeting_only → ALL 格 → corroborate，`sb_only` 一行計數；
每格列填報原文｜會議摘要｜外送版｜理由｜前 N 條證據（摘要、原文、日期、發言方、回連 pk_seq）｜其餘 item_id｜敏感度｜審核欄；
(3) `dry_run = false` 時把關卡結果寫一列 `b_meeting_migration_log`（slug `w2_t5_export:<brief_month>`；D37，不另建 job_run_log）。

**不做什麼**：不打 LLM、不改對照表、不改審核欄。md 只是給人看的產物，正本是表；PM 審完直接 UPDATE 表的
`review_verdict / review_relation / review_note`（md 開頭附 SQL 範例）。

**怎麼跑**：T4 跑完同一期之後。`brief_month` 留空 = 本月。`dry_run = true` 只印不寫 log。印出來的 md 整段複製給 PM。

cell 順序：[c01] 參數 → [c02] 常數 → [c03] 純關卡 → [c04] 純 md 渲染 → [c05] 純 DataFrame 篩選 → [c10] 讀表 → [c11] 算 → [c12] 關卡（硬關卡 assert）→ [c20] 寫 log → [c21] 印 md → [c30] 驗證。

In [ ]:
# [c01] params
# W2 T5：一次匯出一期。由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
import re as _re
from datetime import date as _date

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")                       # g_meeting_sb_reconcile_{unit} / g_meeting_items_sb_{unit}
dbutils.widgets.text("brief_month", "")                  # yyyy-MM；空 = 本月
dbutils.widgets.text("max_evidence", "3")                # 每格列幾條證據（other 格：每個主標籤幾條）
dbutils.widgets.text("max_other_ids", "20")              # 「其餘 item_id」最多列幾個，超過只印 +n
dbutils.widgets.dropdown("include_corroborate", "true", ["true", "false"])  # corroborate 格要不要列（§4.6 排最後；PM 可選不看）
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：只印關卡與 md，不寫 log

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "brief_month": dbutils.widgets.get("brief_month").strip() or _date.today().strftime("%Y-%m"),
    "max_evidence": int(dbutils.widgets.get("max_evidence")),
    "max_other_ids": int(dbutils.widgets.get("max_other_ids")),
    "include_corroborate": dbutils.widgets.get("include_corroborate") == "true",
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert _re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", cfg["brief_month"]), "brief_month 格式 yyyy-MM"
assert 1 <= cfg["max_evidence"] <= 10 and 0 <= cfg["max_other_ids"] <= 200, "max_evidence 1–10；max_other_ids 0–200"
print(cfg)

In [ ]:
# [c02] imports
# [c02]～[c05] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils），tests/ 直接載入。常數對齊 004 [c02] 與 T4 [c02]。
import json
import re
from collections import defaultdict
from datetime import date, datetime

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SLUG = "w2_t5_export"
APPS = ("TV", "NB", "MNT")
FACETS = ("market_outlook", "product_tech", "price", "other")
FACET_LABEL = {"market_outlook": "市場展望", "product_tech": "產品技術", "price": "價格", "other": "其他"}
BRAND_ALL = "ALL"
BRAND_UNKNOWN = "UNKNOWN"
RELATIONS_LLM = ("corroborate", "supplement", "conflict")
RELATIONS = RELATIONS_LLM + ("meeting_only", "sb_only")
# §4.6 清單順序：每 app 一節，conflict → supplement → meeting_only → ALL 格（跨客戶訊號）→ corroborate；sb_only 一行計數
SECTIONS = (("conflict", "衝突：會議端與填報方向不同（先看）"),
            ("supplement", "補充：填報沒提、會議有可行動訊息"),
            ("meeting_only", "只有會議：填報該格空白或該期沒這個品牌"),
            ("app_level", "app 層（ALL 格）：跨客戶共同訊號"),
            ("corroborate", "一致：會議印證填報"))
SIDE_LABEL = {"customer": "客戶方", "auo": "AUO 方", "third": "第三方"}
REVIEW_VERDICTS = ("agree", "fix_relation", "drop")
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")

# 從 v_meeting_sb_reconcile_q 讀的欄
CELL_COLS = ["app_name", "brand", "facet", "brief_month", "window_start", "window_end", "sb_text", "sb_source",
             "meeting_summary", "meeting_summary_shareable", "shareable_failed", "meeting_item_ids", "n_items", "n_items_sent",
             "topic_tags_agg", "relation", "relation_basis", "rationale", "number_caveat", "cross_customer_flag",
             "cross_customer_note", "sensitivity_max", "reviewer", "review_verdict", "review_relation", "review_note",
             "prompt_version", "model", "updated_at", "relation_rank", "effective_relation", "review_pending"]
# 從 v_meeting_items_sb_q 讀的欄（證據用；只取本期 active 且 sb_relevant）
ITEM_COLS = ["item_id", "item_key", "app_name", "brand", "facet", "pk_seq", "segment_no", "sub_no", "event_date",
             "speaker_side", "statement_type", "topic_tag_primary", "summary", "quote", "sensitivity", "subject_clean"]

# 關卡門檻（硬：assert 停；軟：只印 WARN）
GATE_HARD = {
    "rows_zero": "本期對照表 0 格：T4 還沒跑這一期（或 T3 本期沒相關條目也沒填報）",
    "mixed_prompt_version": "本期格子的 prompt_version 不只一種：T4 換版後沒整期重跑",
    "llm_without_summary": "llm 判的格沒有 meeting_summary（T4 [c30] 應該已擋）",
    "bad_relation": "relation 不在五種合法值內",
}
GATE_SOFT = {
    "meeting_only_without_summary": "meeting_only 格沒摘要（T4 摘要批失敗或被 max_cells 跳過）：清單只列條目、沒有摘要",
    "shareable_failed_rate": 0.2,     # 外送版驗證失敗比例 > 20%：prompt 的 S2 規則要調
    "items_not_in_cells": "T3 本期相關條目數 ≠ 品牌格 n_items 合計：T3 在 T4 之後又重標過，T4 要重跑",
    "no_sb_text": "本期沒有任何填報文字：brief 還沒填，全部 meeting_only（預期內，brief 填完後 T3 → T4 → T5 重跑）",
}

In [ ]:
# [c03] pure_gates
# 對照格清單（dict）＋ T3 本期統計 → 關卡彙總 dict ＋ 硬關卡 / 軟關卡訊息。純 Python，不碰 Spark。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


def _dist(rows: list[dict], key: str) -> dict:
    d: dict = defaultdict(int)
    for r in rows:
        d[r.get(key) or "NULL"] += 1
    return dict(sorted(d.items(), key=lambda kv: (-kv[1], str(kv[0]))))


def summarize_cells(cells: list[dict], label_stats: dict | None = None) -> dict:
    """cells = v_meeting_sb_reconcile_q 本期的列（dict）。label_stats = {"relevant_items": n, "label_rows": n}（T3 本期）。"""
    brand = [c for c in cells if c.get("brand") != BRAND_ALL]
    app = [c for c in cells if c.get("brand") == BRAND_ALL]
    llm = [c for c in cells if c.get("relation_basis") == "llm"]
    with_summary = [c for c in cells if c.get("meeting_summary")]
    s = {
        "brief_month": cells[0].get("brief_month") if cells else None,
        "cells": len(cells), "brand_cells": len(brand), "app_cells": len(app),
        "relation_dist": _dist(cells, "relation"),
        "llm_cells": len(llm),
        "llm_without_summary": sum(1 for c in llm if not c.get("meeting_summary")),
        "meeting_only_without_summary": sum(1 for c in cells if c.get("relation") == "meeting_only" and not c.get("meeting_summary")),
        "bad_relation": sum(1 for c in cells if c.get("relation") not in RELATIONS),
        "shareable_failed": sum(1 for c in cells if c.get("shareable_failed")),
        "shareable_failed_rate": pct(sum(1 for c in cells if c.get("shareable_failed")), len(with_summary)),
        "cross_customer_flagged": sum(1 for c in brand if c.get("cross_customer_flag")),
        "signals": sum(1 for c in app if c.get("cross_customer_note")),
        "review_pending": sum(1 for c in cells if c.get("review_pending")),
        "reviewed": sum(1 for c in cells if c.get("review_verdict")),
        "review_verdict_dist": _dist(cells, "review_verdict"),
        "prompt_versions": sorted({c.get("prompt_version") for c in cells if c.get("prompt_version")}),
        "models": sorted({c.get("model") for c in cells if c.get("model")}),
        "sensitivity_dist": _dist(cells, "sensitivity_max"),
        "n_items_brand_cells": sum(int(c.get("n_items") or 0) for c in brand),
        "has_sb_text": any(c.get("sb_text") for c in cells),
        "unknown_brand_cells": sum(1 for c in brand if c.get("brand") == BRAND_UNKNOWN),
        "last_updated_at": max((c.get("updated_at") for c in cells if c.get("updated_at")), default=None),
    }
    if label_stats:
        s["label_relevant_items"] = label_stats.get("relevant_items")
        s["label_rows"] = label_stats.get("label_rows")
    return s


def run_gates(s: dict) -> tuple[list[str], list[str]]:
    """彙總 → (硬關卡失敗訊息, 軟關卡 WARN)。硬關卡任一失敗 = task fail（[c12] assert）。"""
    hard, soft = [], []
    if s["cells"] == 0:
        hard.append(GATE_HARD["rows_zero"])
        return hard, soft
    if len(s["prompt_versions"]) > 1:
        hard.append(f"{GATE_HARD['mixed_prompt_version']}：{s['prompt_versions']}")
    if s["llm_without_summary"]:
        hard.append(f"{GATE_HARD['llm_without_summary']}：{s['llm_without_summary']} 格")
    if s["bad_relation"]:
        hard.append(f"{GATE_HARD['bad_relation']}：{s['bad_relation']} 格")
    if s["meeting_only_without_summary"]:
        soft.append(f"{GATE_SOFT['meeting_only_without_summary']}：{s['meeting_only_without_summary']} 格")
    if (s["shareable_failed_rate"] or 0) > GATE_SOFT["shareable_failed_rate"]:
        soft.append(f"外送版驗證失敗 {s['shareable_failed']} 格（{s['shareable_failed_rate']:.0%} > "
                    f"{GATE_SOFT['shareable_failed_rate']:.0%}）：T4 [c04] 的 S2 規則要調")
    if s.get("label_relevant_items") is not None and s["label_relevant_items"] != s["n_items_brand_cells"]:
        soft.append(f"{GATE_SOFT['items_not_in_cells']}：T3 相關 {s['label_relevant_items']} vs 格內 {s['n_items_brand_cells']}")
    if not s["has_sb_text"]:
        soft.append(GATE_SOFT["no_sb_text"])
    return hard, soft

In [ ]:
# [c04] pure_render
# 對照格 ＋ 證據條目 → PM 例外清單 md（規格 §4.6）。md 是給人看 / 貼給 PM 的產物，正本是表。
# 不出現姓名 / email：條目本來就 token 化（<Pnn>），這裡再把 email 樣式換 <email>。
def _c(v) -> str:
    """md 單行文字：None → ''、壓平空白、原文的 | 換成 ／（清單不是表格，不用跳脫；原文的 | 多是表格殘留）、email 遮罩。"""
    s = re.sub(r"\s+", " ", ("" if v is None else str(v)).replace("\xa0", " ")).strip()
    return EMAIL_ANY_RE.sub("<email>", s).replace("|", "／")


def _d(v) -> str:
    return v.isoformat() if isinstance(v, (date, datetime)) else ("" if v is None else str(v))


def side_label(side: str | None) -> str:
    return SIDE_LABEL.get((side or "").lower(), "與會者")


def cell_title(c: dict) -> str:
    brand = c.get("brand")
    brand_txt = "ALL（app 層）" if brand == BRAND_ALL else ("UNKNOWN（客戶不明）" if brand == BRAND_UNKNOWN else brand)
    return f"{c.get('app_name')} × {brand_txt} × {FACET_LABEL.get(c.get('facet'), c.get('facet'))}"


def section_of(c: dict) -> str | None:
    """格 → §4.6 的節：ALL 格一律 app_level；sb_only 不列（只計數）；其餘照 effective_relation（PM 改過就用改過的）。"""
    if c.get("brand") == BRAND_ALL:
        return "app_level"
    rel = c.get("effective_relation") or c.get("relation")
    return rel if rel in ("conflict", "supplement", "meeting_only", "corroborate") else None


def pick_evidence(c: dict, items_by_id: dict, max_evidence: int) -> tuple[list, list[str], list[str]]:
    """meeting_item_ids（證據排前面）→ (顯示用 [(tag, item)...], 其餘 item_id, 找不到的 id)。
    other 格每個 topic_tag_primary 最多 max_evidence 條（§4.6：other 以標籤分組）；其餘面向整格最多 max_evidence 條。"""
    ids = [x for x in (c.get("meeting_item_ids") or []) if x]
    found = [(i, items_by_id[i]) for i in ids if i in items_by_id]
    missing = [i for i in ids if i not in items_by_id]
    shown: list = []
    if c.get("facet") == "other":
        per_tag: dict = defaultdict(int)
        for i, it in found:
            tag = it.get("topic_tag_primary") or "（無標籤）"
            if per_tag[tag] < max_evidence:
                per_tag[tag] += 1
                shown.append((tag, i, it))
        order = {t: n for n, t in enumerate(dict.fromkeys(t for t, _i, _it in shown))}  # 標籤照首次出現排，同標籤集中
        shown.sort(key=lambda x: order[x[0]])
    else:
        shown = [(None, i, it) for i, it in found[:max_evidence]]
    shown_ids = {i for _, i, _ in shown}
    rest = [i for i in ids if i not in shown_ids]
    return shown, rest, missing


def _evidence_line(it: dict) -> str:
    where = f"pk {it.get('pk_seq')}" + (f" seg {it.get('segment_no')}" if it.get("segment_no") is not None else "")
    quote = _c(it.get("quote"))
    quote = f"「{quote[:200]}{'…' if len(quote) > 200 else ''}」" if quote else ""
    return (f"{_d(it.get('event_date'))}｜{side_label(it.get('speaker_side'))}｜{_c(it.get('statement_type'))}/"
            f"{_c(it.get('topic_tag_primary'))}｜{_c(it.get('sensitivity'))}｜{_c(it.get('summary'))}｜{quote}｜{where}")


def render_cell(c: dict, items_by_id: dict, max_evidence: int, max_other_ids: int, flagged_brands: list[str] | None = None) -> list[str]:
    rel = c.get("relation")
    eff = c.get("effective_relation") or rel
    rel_txt = rel if eff == rel else f"{rel} → PM 改 {eff}"
    out = [f"#### {cell_title(c)} ｜ {rel_txt}（{_c(c.get('relation_basis'))}）｜ {_c(c.get('sensitivity_max')) or 'S?'}"
           f" ｜ 條目 {c.get('n_items') or 0}（送 LLM {c.get('n_items_sent') or 0}）｜ 窗口 {_d(c.get('window_start'))}～{_d(c.get('window_end'))}"]
    out.append(f"- 填報：{_c(c.get('sb_text')) or '（空白）'}")
    out.append(f"- 會議：{_c(c.get('meeting_summary')) or '（無摘要）'}")
    if c.get("shareable_failed"):
        out.append("- 外送版：（驗證失敗：含 S2 數字，未產；要外送請 PM 自己寫）")
    elif c.get("meeting_summary_shareable"):
        out.append(f"- 外送版：{_c(c.get('meeting_summary_shareable'))}")
    if c.get("rationale") or c.get("number_caveat"):
        parts = [p for p in (("理由：" + _c(c["rationale"])) if c.get("rationale") else None,
                             ("數字口徑：" + _c(c["number_caveat"])) if c.get("number_caveat") else None) if p]
        out.append("- " + " ｜ ".join(parts))
    if c.get("brand") == BRAND_ALL:
        out.append(f"- 跨客戶訊號：{_c(c.get('cross_customer_note')) or '（無 ≥ 2 品牌共同訊號）'}")
        if flagged_brands:
            out.append(f"- 參與品牌格：{', '.join(flagged_brands)}")
    shown, rest, missing = pick_evidence(c, items_by_id, max_evidence)
    if shown:
        if c.get("facet") == "other":
            out.append("- 證據（依主標籤分組）：")
            cur = None
            for tag, _i, it in shown:
                if tag != cur:
                    out.append(f"  - {tag}：")
                    cur = tag
                out.append(f"    - {_evidence_line(it)}")
        else:
            out.append("- 證據：")
            out += [f"  - {_evidence_line(it)}" for _t, _i, it in shown]
    if rest:
        head = rest[:max_other_ids]
        more = f" …（+{len(rest) - len(head)}）" if len(rest) > len(head) else ""
        out.append(f"- 其餘條目：{', '.join(head)}{more}")
    if missing:
        out.append(f"- （{len(missing)} 條證據 id 已不在本期標註表：T3 重標過，T4 要重跑）")
    if c.get("review_verdict"):
        out.append(f"- 審核：{_c(c['review_verdict'])}"
                   + (f" → {_c(c['review_relation'])}" if c.get("review_relation") else "")
                   + (f"｜{_c(c['review_note'])}" if c.get("review_note") else "")
                   + (f"｜{_c(c['reviewer'])}" if c.get("reviewer") else ""))
    else:
        out.append("- 審核：待審")
    out.append("")
    return out


def _review_sql(fq_table: str) -> list[str]:
    return ["```sql", f"-- PM 審核：直接更新對照表（T4 重跑不會覆蓋這五欄）",
            f"UPDATE {fq_table}", "SET reviewer = '<你的代號>', review_verdict = 'agree',          -- agree / fix_relation / drop",
            "    review_relation = NULL,                                 -- fix_relation 時填正確 relation",
            "    review_note = NULL",
            "WHERE brief_month = '<yyyy-MM>' AND app_name = '<app>' AND brand = '<brand>' AND facet = '<facet>';", "```"]


def render_review_md(cells: list[dict], items_by_id: dict, summary: dict, warns: list[str], *, max_evidence: int = 3,
                     max_other_ids: int = 20, include_corroborate: bool = True, fq_table: str = "<catalog>.<schema>.g_meeting_sb_reconcile_bu",
                     generated_at: str | None = None) -> str:
    """§4.6 例外清單。每 app 一節；節內 conflict → supplement → meeting_only → ALL 格 → corroborate；sb_only 一行計數。
    同節內品牌格依 n_items 降冪；UNKNOWN 品牌格排該節最後。"""
    bm = summary.get("brief_month") or "?"
    out = [f"# Sales Brief 對照例外清單 — {bm}", "",
           f"由 `meeting_extracted_w2_t5_export` 產生{('（' + generated_at + '）') if generated_at else ''}；正本是 `{fq_table}`，"
           f"T4 prompt {', '.join(summary.get('prompt_versions') or ['?'])}。",
           f"格數 {summary.get('cells', 0)}（品牌格 {summary.get('brand_cells', 0)} ＋ app 層 {summary.get('app_cells', 0)}）："
           + "、".join(f"{k} {v}" for k, v in (summary.get("relation_dist") or {}).items())
           + f"；待審 {summary.get('review_pending', 0)}、已審 {summary.get('reviewed', 0)}。", ""]
    if warns:
        out += ["> ⚠ " + w for w in warns] + [""]
    out += ["**怎麼審**：每格看「填報 vs 會議」與證據，三選一填 `review_verdict`：`agree`（判得對）/ `fix_relation`（關係判錯，`review_relation` 填正確的）/ "
            "`drop`（這格不該出現：條目不相關或重複）。`review_note` 隨意。會議摘要含 S2 細節只給內部；要外送用「外送版」。人以 `<Pnn>` token 顯示，不還原姓名。", ""]
    out += _review_sql(fq_table) + [""]
    by_app: dict = defaultdict(list)
    for c in cells:
        by_app[c.get("app_name") or "?"].append(c)
    apps = [a for a in APPS if a in by_app] + sorted(a for a in by_app if a not in APPS)
    for app in apps:
        cs = by_app[app]
        out += [f"## {app}", ""]
        sb_only = [c for c in cs if c.get("brand") != BRAND_ALL and (c.get("effective_relation") or c.get("relation")) == "sb_only"]
        groups: dict = defaultdict(list)
        for c in cs:
            sec = section_of(c)
            if sec:
                groups[sec].append(c)
        flagged_by_facet: dict = defaultdict(list)
        for c in cs:
            if c.get("brand") != BRAND_ALL and c.get("cross_customer_flag"):
                flagged_by_facet[c.get("facet")].append(c.get("brand"))
        n = 0
        for sec, label in SECTIONS:
            if sec == "corroborate" and not include_corroborate:
                continue
            g = groups.get(sec) or []
            n += 1
            out += [f"### {n}. {label}（{len(g)} 格）", ""]
            if not g:
                out += ["（無）", ""]
                continue
            g = sorted(g, key=lambda c: (c.get("brand") == BRAND_UNKNOWN, -(c.get("n_items") or 0), c.get("brand") or "", c.get("facet") or ""))
            for c in g:
                out += render_cell(c, items_by_id, max_evidence, max_other_ids,
                                   flagged_brands=sorted(set(flagged_by_facet.get(c.get("facet"), []))) if sec == "app_level" else None)
        brands = sorted({c.get("brand") for c in sb_only if c.get("brand")})
        out += [f"- sb_only（只有填報、本期沒有相關會議條目）：{len(sb_only)} 格" + (f"（{', '.join(brands)}）" if brands else ""), ""]
    return "\n".join(out) + "\n"

In [ ]:
# [c05] pure_spark
# DataFrame -> DataFrame 的篩選（先過濾再 select；本期的格與條目都只有幾百列，collect 到 driver 做 md）。
def select_month_cells(df_rec_q: DataFrame, brief_month: str) -> DataFrame:
    """v_meeting_sb_reconcile_q → 本期格（CELL_COLS）。"""
    return df_rec_q.where(F.col("brief_month") == F.lit(brief_month)).select(*CELL_COLS)


def select_month_items(df_items_q: DataFrame, brief_month: str) -> DataFrame:
    """v_meeting_items_sb_q → 本期 active 且 sb_relevant 的條目（證據用，ITEM_COLS）。同一 item_id 可能因 app 展開成多列，證據只要一列。"""
    return (df_items_q.where((F.col("brief_month") == F.lit(brief_month)) & F.col("active") & F.col("sb_relevant"))
            .select(*ITEM_COLS).dropDuplicates(["item_id"]))


def label_stats_df(df_items_q: DataFrame, brief_month: str) -> DataFrame:
    """T3 本期統計：標註列數、相關條目數（(item_key, app_name) 列；相關條目 = sb_relevant 列，和 T4 品牌格 n_items 合計同口徑）。"""
    return (df_items_q.where((F.col("brief_month") == F.lit(brief_month)) & F.col("active"))
            .agg(F.count("*").alias("label_rows"), F.sum(F.col("sb_relevant").cast("int")).alias("relevant_items")))

In [ ]:
# [c10] load_source
# 讀本期對照格（View 已帶 relation_rank / effective_relation / review_pending）、本期相關條目（證據）、T3 本期統計 → collect 到 driver。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_rec_q, fq_items_q = fq("v_meeting_sb_reconcile_q"), fq("v_meeting_items_sb_q")
fq_rec, fq_log = fq(f"g_meeting_sb_reconcile_{cfg['unit']}"), fq("b_meeting_migration_log")
for t in (fq_rec_q, fq_items_q, fq_rec, fq_log):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 004 與 T3 / T4"

bm = cfg["brief_month"]
cells = [r.asDict() for r in select_month_cells(spark.table(fq_rec_q), bm).collect()]
items = [r.asDict() for r in select_month_items(spark.table(fq_items_q), bm).collect()]
items_by_id = {it["item_id"]: it for it in items}
ls = label_stats_df(spark.table(fq_items_q), bm).first()
label_stats = {"label_rows": int(ls["label_rows"] or 0), "relevant_items": int(ls["relevant_items"] or 0)}
print(f"brief_month {bm} | cells {len(cells)} | evidence items {len(items)} | T3 本期 {label_stats}")

In [ ]:
# [c11] run_export
# 純函式串起來：彙總 → 關卡 → md。
summary = summarize_cells(cells, label_stats)
hard_fails, warns = run_gates(summary)
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")
md = render_review_md(cells, items_by_id, summary, warns, max_evidence=cfg["max_evidence"], max_other_ids=cfg["max_other_ids"],
                      include_corroborate=cfg["include_corroborate"], fq_table=fq_rec, generated_at=generated_at)
print(f"md {len(md):,} 字 | hard {len(hard_fails)} | warn {len(warns)}")

In [ ]:
# [c12] gates
# 印彙總 JSON；軟關卡只 WARN；硬關卡（0 格 / prompt 版本混雜 / llm 格沒摘要 / relation 不合法）assert 停，後面不寫 log、不印 md。
print(json.dumps(summary, ensure_ascii=False, default=str, indent=1))
for w in warns:
    print("WARN", w)
for h in hard_fails:
    print("FAIL", h)
assert not hard_fails, f"T5 硬關卡失敗 {len(hard_fails)} 項（見上方 FAIL）：先修 T4 這一期再匯出"
if cfg["dry_run"]:
    print("dry_run = true：不寫 log；md 在 [c21] 印出")

In [ ]:
# [c20] write_log
# 關卡結果寫一列 b_meeting_migration_log（slug w2_t5_export:<brief_month>；只 INSERT，不改舊列；D37 不另建 job_run_log）。
log_slug = f"{SLUG}:{bm}"
if not cfg["dry_run"]:
    result = {"summary": summary, "warns": warns, "cfg": {k: v for k, v in cfg.items() if k not in ("catalog", "schema")},
              "md_chars": len(md)}
    spark.sql(
        f"INSERT INTO {fq_log} SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": log_slug, "result": json.dumps(result, ensure_ascii=False, default=str), "note": f"gate ok; warns {len(warns)}"},
    )
    print("logged", log_slug)
else:
    print("skip log（dry_run）")

In [ ]:
# [c21] print_md
# 整份 md 印出來（PM 例外清單；已 token 化、無人名）；使用者整段複製貼給 PM。正本是對照表，md 不落地。
print(md)

In [ ]:
# [c30] check_target
# dry_run = false：log 表最新一列 slug 對、result 解得開、cells 數和 [c11] 一致；md 不含 email 樣式。
assert not EMAIL_ANY_RE.search(md), "md 含 email 樣式（S3 規則）：看 [c04] _c 遮罩"
if not cfg["dry_run"]:
    last = (spark.table(fq_log).where(F.col("slug") == F.lit(log_slug)).orderBy(F.desc("run_at"))
            .select("run_at", "run_by", "result").limit(1).collect())
    assert last, f"{fq_log} 沒有 {log_slug} 的列"
    got = json.loads(last[0]["result"])
    chk = {"run_at": str(last[0]["run_at"]), "cells_logged": got["summary"]["cells"], "cells_expected": summary["cells"],
           "warns_logged": len(got["warns"])}
    print(chk)
    assert chk["cells_logged"] == chk["cells_expected"], "log 的格數和本次不一致"